<a href="https://www.kaggle.com/code/lalit7881/predicting-customer-churn-using-ml?scriptVersionId=305273782" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


## Loading dataset

In [2]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv') 

## Basic Inspection

In [3]:
print(train.shape, test.shape)

train.head()
train.info()
train.describe()

(594194, 21) (254655, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 

,id,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,0.114102,36.577258,65.866223,2494.377057
std,171529.177262,0.317936,25.061922,31.067444,2353.916710
min,0.000000,0.000000,1.000000,18.250000,18.800000
25%,148548.250000,0.000000,12.000000,29.900000,639.650000
50%,297096.500000,0.000000,35.000000,74.100000,1433.650000
75%,445644.750000,0.000000,62.000000,90.800000,4263.800000
max,594193.000000,1.000000,72.000000,118.750000,8684.800000


## Check Missing Values

In [4]:
train.isnull().sum().sort_values(ascending=False)
test.isnull().sum().sort_values(ascending=False)

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

In [5]:
print(train.info())
print(train.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

## Handle Duplicates

In [6]:
train.drop_duplicates(inplace=True)

## Feature engineering

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [8]:
train["Churn"] = train["Churn"].map({"Yes": 1, "No": 0})

for df in [train, test]:
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')
    df.fillna(0, inplace=True)

In [9]:
for df in [train, test]:
    df["AvgCharge"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

services = [
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]

for df in [train, test]:
    df["NumServices"] = (
        (df[services] != "No") &
        (df[services] != "No internet service") &
        (df[services] != "No phone service")
    ).sum(axis=1)

In [10]:
combined = pd.concat([train.drop("Churn", axis=1), test])
combined = pd.get_dummies(combined, drop_first=True)

X = combined.iloc[:len(train), :]
X_test = combined.iloc[len(train):, :]
y = train["Churn"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train XGBoost
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

# Validate
xgb_val_preds = xgb_model.predict_proba(X_val)[:, 1]
print("XGB ROC-AUC:", roc_auc_score(y_val, xgb_val_preds))

XGB ROC-AUC: 0.9166196414805926


In [11]:
xgb_model.fit(X, y)
xgb_test_preds = xgb_model.predict_proba(X_test)[:, 1]

In [12]:
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()

X_cat = train.drop("Churn", axis=1)
y_cat = train["Churn"]

X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
    X_cat, y_cat, test_size=0.2, random_state=42
)

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.15,
    depth=5,
    eval_metric='AUC',
    random_seed=42,
    verbose=0
)

cat_model.fit(
    X_train_c, y_train_c,
    cat_features=cat_cols,
    eval_set=(X_val_c, y_val_c)
)

# Validate
cat_val_preds = cat_model.predict_proba(X_val_c)[:, 1]
print("CAT ROC-AUC:", roc_auc_score(y_val_c, cat_val_preds))


CAT ROC-AUC: 0.9153407387984902


In [13]:
cat_model.fit(X_cat, y_cat, cat_features=cat_cols)
cat_test_preds = cat_model.predict_proba(test)[:, 1]


In [14]:
final_test_preds = (
    0.5 * xgb_test_preds +
    0.5 * cat_test_preds
)

In [15]:
# Load data
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

# Save IDs
test_ids = test["id"]

# (your preprocessing, feature engineering, models...)

# Final submission
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": final_test_preds
})

submission.to_csv("final_submission.csv", index=False)

## Thank you..pls upvote!!!!!!!!